In [4]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.preprocessing.split_dataset import split_dataset
from src.preprocessing.build_interactions import filter_interactions

N_NEGATIVES = 100

# Load from raw interactions 

In [5]:
df = pd.read_csv("../data/raw/interactions_v03.csv")

# remove `ent_rate`, `time_type`, `group` columns
df = df.drop(columns=["ent_rate", "time_type", "group"])

df = df.rename(columns={"pfid": "user_id", "anchor_id": "streamer_id"})

print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"Number of unique users: {df['user_id'].nunique()}")
print(f"Number of unique streamers: {df['streamer_id'].nunique()}")

Number of rows: 4587089
Number of columns: 8
Number of unique users: 90183
Number of unique streamers: 6486


In [153]:
filters = [
    lambda df: (df["consume_cnt"] > 0) | (df["prod_total"] > 0),  # keep entries with donations or products
]

df = filter_interactions(df, filters=filters)
print(f"Number of rows after filtering donations/products: {df.shape[0]}")

# # deduplicate interactions
# df = df.groupby(["user_id", "streamer_id"], as_index=False).first()
# print(f"Number of rows after deduplication: {df.shape[0]}")

Filter 1: filtered out 4390079 rows
Total filtered: 4390079 (from 4587089 to 197010)
Number of rows after filtering donations/products: 197010


In [154]:
STREAMER_ID_COL = "streamer_id"

# filter out interactions with streamers not in the embedding lookup
streamer_lookup = "../embeddings/MiniLM/format_sentence/lookup.parquet"
streamer_ids_with_embeddings = set(pd.read_parquet(streamer_lookup)["streamer_id"].to_list())

before = len(df)
df = df[df[STREAMER_ID_COL].isin(streamer_ids_with_embeddings)]
after = len(df)
print(f"Filtered out {before - after} interactions with streamers not in the embedding lookup ({before} → {after})")

Filtered out 9164 interactions with streamers not in the embedding lookup (197010 → 187846)


## Split train, val, test interactions

split user's interacted streamer that should be in training, validation, and testing set

In [155]:
# train_df: all positive interactions
# val_df: 1 positive, N negatives
# test_df: 1 positive, N negatives

train_df, val_df, test_df = split_dataset(df, neg_per_pos=N_NEGATIVES)
train_df.shape, val_df.shape, test_df.shape

((152109, 3), (975761, 3), (1297850, 3))

# Load from pre-processed

In [41]:
train_df = pd.read_parquet("../data/splits/donate/train.parquet")
val_df = pd.read_parquet("../data/splits/donate/val.parquet")
test_df = pd.read_parquet("../data/splits/donate/test.parquet")

train_df.shape, val_df.shape, test_df.shape

((152109, 3), (975761, 3), (1297850, 3))

# Feature engineering

build user & item aggregated features, then reapply it onto train, val, test set

In [42]:
# filter interactions for (user_id, streamer_id) that appears in train set
train_interaction_logs = df.merge(
    train_df[["user_id", "streamer_id"]],
    on=["user_id", "streamer_id"],
    how="inner"
)

In [43]:
train_interaction_logs.shape

(255240, 8)

In [44]:
# build user & item aggregated features, then reapply it onto train, val, test set
user_df = (
    train_interaction_logs.groupby("user_id").agg(
        u_watch_tot=("watch_ts", "sum"),
        u_watch_cnt=("watch_ts", "size"),
        u_gift_cnt =("consume_cnt", "sum"),
        u_gift_amt =("prod_total", "sum"),
        u_follow_cnt=("is_follow", "sum"),
    )
)

item_df = (
    train_interaction_logs.groupby("streamer_id").agg(
        i_watch_tot=("watch_ts", "sum"),
        i_watch_cnt=("watch_ts", "size"),
        i_unique_user=("user_id", "nunique"),
        i_live_cnt  =("live_cnt", "max"),      # already monthly total
        i_followers =("is_follow", "sum"),
        i_gift_amt  =("prod_total", "sum"),
    )
    .assign(
        i_watch_avg = lambda d: d.i_watch_tot / d.i_watch_cnt,
        i_pop_z     = lambda d: ((d.i_watch_cnt - d.i_watch_cnt.mean()) 
                                 / d.i_watch_cnt.std()),
    )
)

In [45]:
# sample N negatives for each user in train_df
# all_streamers = set(df["streamer_id"].unique())
streamers_in_train = set(train_df["streamer_id"].unique())
neg_samples = []

for user_id, group in train_df.groupby("user_id"):
    pos_streamers = set(group["streamer_id"])
    neg_candidates = list(streamers_in_train - pos_streamers)
    if len(neg_candidates) < N_NEGATIVES:
        sampled_negs = neg_candidates  # take all if not enough
    else:
        sampled_negs = np.random.choice(neg_candidates, N_NEGATIVES, replace=False)
    for neg in sampled_negs:
        neg_samples.append({"user_id": user_id, "streamer_id": neg, "label": 0})

neg_df = pd.DataFrame(neg_samples)
train_with_negs = pd.concat([train_df, neg_df], ignore_index=True)

In [46]:
train_with_negs

,user_id,streamer_id,label
0,1000015,5734817,1
1,1000015,3790558,1
2,1000015,1181624,1
3,1000015,6792883,1
4,1000015,6209326,1
...,...,...,...
1437104,6833170,3182483,0
1437105,6833170,6486271,0
1437106,6833170,1311300,0
1437107,6833170,3733093,0


In [27]:
# reapply user & item aggregated features onto train, val, test set
train_with_negs = train_with_negs.merge(user_df, on="user_id", how="left")
train_with_negs = train_with_negs.merge(item_df, on="streamer_id", how="left")

val_df = val_df.merge(user_df, on="user_id", how="left")
val_df = val_df.merge(item_df, on="streamer_id", how="left")

test_df = test_df.merge(user_df, on="user_id", how="left")
test_df = test_df.merge(item_df, on="streamer_id", how="left")

# fill NaN for cold-start streamers
val_df = val_df.fillna(val_df.mean())
test_df = test_df.fillna(test_df.mean())

In [28]:
train_with_negs

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z
0,1000015,5734817,1,555813.0,28,534.0,782972.0,5.0,2560913.0,165,101,44.0,13.0,1329533.0,15520.684848,0.046898
1,1000015,3790558,1,555813.0,28,534.0,782972.0,5.0,2734612.0,177,113,42.0,23.0,3612331.0,15449.785311,0.121521
2,1000015,1181624,1,555813.0,28,534.0,782972.0,5.0,1429111.0,140,88,36.0,6.0,165091.0,10207.935714,-0.108566
3,1000015,6792883,1,555813.0,28,534.0,782972.0,5.0,6812359.0,1431,839,41.0,210.0,1189807.0,4760.558351,7.919580
4,1000015,6209326,1,555813.0,28,534.0,782972.0,5.0,5463315.0,218,138,54.0,16.0,2421403.0,25061.077982,0.376481
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1437104,6833170,5331064,0,113.0,2,1.0,2.0,1.0,1833083.0,151,92,34.0,9.0,880392.0,12139.622517,-0.040162
1437105,6833170,5752534,0,113.0,2,1.0,2.0,1.0,455133.0,79,44,16.0,1.0,215771.0,5761.177215,-0.487897
1437106,6833170,6819671,0,113.0,2,1.0,2.0,1.0,6055.0,15,9,23.0,5.0,562.0,403.666667,-0.885884
1437107,6833170,6823536,0,113.0,2,1.0,2.0,1.0,13837.0,58,36,14.0,6.0,1082.0,238.568966,-0.618487


In [160]:
train_with_negs.columns

Index(['user_id', 'streamer_id', 'label', 'u_watch_tot', 'u_watch_cnt',
       'u_gift_cnt', 'u_gift_amt', 'u_follow_cnt', 'i_watch_tot',
       'i_watch_cnt', 'i_unique_user', 'i_live_cnt', 'i_followers',
       'i_gift_amt', 'i_watch_avg', 'i_pop_z'],
      dtype='object')

In [161]:
FEATURE_COLS = [
    # user features
    'u_watch_tot', 'u_watch_cnt', 'u_gift_cnt', 'u_gift_amt', 'u_follow_cnt', 
    # item features
    'i_watch_tot', 'i_watch_cnt', 'i_unique_user', 'i_live_cnt', 'i_followers', 'i_gift_amt', 'i_watch_avg', 'i_pop_z'
]

In [162]:
bad_users = train_with_negs.groupby('user_id').label.nunique()
assert (bad_users == 2).all(), "some users miss pos or neg"

# Train LightGBMRanker

In [163]:
from lightgbm import LGBMRanker, early_stopping
from sklearn.preprocessing import StandardScaler

In [164]:
train_with_negs.head()

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z
0,1000015,6758324,1,0.0,14,534.0,782972.0,5.0,21195.0,118,118,15.0,6.0,3338.0,179.618644,0.250010
1,1000015,4900393,1,0.0,14,534.0,782972.0,5.0,1033071.0,163,163,41.0,32.0,1499313.0,6337.858896,0.721270
2,1000015,4913097,1,0.0,14,534.0,782972.0,5.0,156624.0,63,63,39.0,17.0,884822.0,2486.095238,-0.325974
3,1000015,6081991,1,0.0,14,534.0,782972.0,5.0,4348078.0,299,299,92.0,26.0,6648869.0,14542.066890,2.145523
4,1000015,1181624,1,0.0,14,534.0,782972.0,5.0,502593.0,86,86,36.0,6.0,161088.0,5844.104651,-0.085108


In [165]:
# Sort rows so group order = row order
train_with_negs.sort_values('user_id', inplace=True)
val_df.sort_values('user_id', inplace=True)

# scale numeric columns (tree prefers but not mandatory)
scaler = StandardScaler().fit(train_with_negs[FEATURE_COLS])

X_train = scaler.transform(train_with_negs[FEATURE_COLS])
y_train = train_with_negs["label"].values

X_val = scaler.transform(val_df[FEATURE_COLS])
y_val = val_df["label"].values

X_test = scaler.transform(test_df[FEATURE_COLS])
y_test = test_df["label"].values

train_group = train_with_negs.groupby("user_id").size().to_numpy()
val_group = val_df.groupby("user_id").size().to_numpy()

In [175]:
train_with_negs.shape

(1437109, 16)

In [170]:
X_test.shape

(1297850, 13)

In [166]:
X_train.shape

(1437109, 13)

In [167]:
ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[10,20,50],
    num_leaves=63,
    n_estimators=100,
    learning_rate=0.05,
)

callbacks = [early_stopping(stopping_rounds=30, verbose=True)]

ranker.fit(
    X_train, y_train,
    group=train_group,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group], 
    callbacks=callbacks,
)

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010065 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2781
[LightGBM] [Info] Number of data points in the train set: 1437109, number of used features: 13
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[94]	valid_0's ndcg@10: 0.201845	valid_0's ndcg@20: 0.244175	valid_0's ndcg@50: 0.304718


LGBMRanker(learning_rate=0.05, metric='ndcg', ndcg_eval_at=[10, 20, 50],
           num_leaves=63, objective='lambdarank')

# Evaluation

In [168]:
scores = ranker.predict(X_test)
test_df["score"] = scores

# split rows by user
sizes  = test_df.groupby("user_id").size().to_numpy() # group vector
idx    = np.cumsum(sizes)[:-1] # cumulative sume, indicating where each group ends
groups = np.split(test_df.to_numpy(), idx) # list of arrays

# col indices for speed
LABEL_COL  = test_df.columns.get_loc("label")
SCORE_COL  = test_df.columns.get_loc("score")

ranks = [] # # rank (1-based) of the positive per user
for group in groups:
    # sort descending by score inside this user group
    group_sorted = group[np.argsort(-group[:, SCORE_COL])]
    # index of the unique positive (label=1) in the sorted group
    pos_rank = np.where(group_sorted[:, LABEL_COL] == 1)[0][0] + 1
    ranks.append(pos_rank)  # store the rank (1-based)
ranks = np.asarray(ranks) # shape [n_users]

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [169]:
# precision is not needed since the denominator is the same with recall in leave-one-out setting

def recall_at_k(r, k):
    return (r <= k).mean()

def ndcg_at_k(r, k):
    return np.where(r <= k, 1 / np.log2(r + 1), 0).mean()

def mrr_at_k(r, k):
    return np.where(r <= k, 1 / r, 0).mean()

KS = (10, 20, 50, 100)
for k in KS:
    print(f"k={k:<2}  "
          f"Recall(Hit rate) {recall_at_k(ranks,k):.4f}  "
          f"nDCG {ndcg_at_k(ranks,k):.4f}  "
          f"MRR  {mrr_at_k(ranks,k):.4f}")

k=10  Recall(Hit rate) 0.3547  nDCG 0.1985  MRR  0.1513
k=20  Recall(Hit rate) 0.5242  nDCG 0.2411  MRR  0.1628
k=50  Recall(Hit rate) 0.8385  nDCG 0.3033  MRR  0.1728
k=100  Recall(Hit rate) 0.9996  nDCG 0.3301  MRR  0.1753


# Analysis

Model is performing too well: possible issues:
* negatives are too easy (streamer that user has never interacted with)

In [90]:
print("rows:", len(ranks))
print("min rank of positive:", ranks.min(), "max rank:", ranks.max())

rows: 12850
min rank of positive: 1 max rank: 101


## Inspect top features

Seems to recommend the most popular streamer

In [91]:
imp = ranker.booster_.feature_importance(importance_type="gain")
for feat, gain in sorted(zip(FEATURE_COLS, imp), key=lambda x: -x[1])[:10]:
    print(feat, gain)

i_watch_cnt 140411.0697145462
i_gift_amt 38344.086950302124
u_watch_cnt 31709.028387069702
u_gift_amt 30882.670128822327
i_watch_avg 17973.98365688324
u_gift_cnt 17614.886658668518
i_live_cnt 13428.128586292267
i_followers 11087.123278617859
u_watch_tot 10109.38229751587
u_follow_cnt 7370.481549739838


## Hold-out cold-item split

a special test set where all positives are streamers that never appear in training. 

In [92]:
# ids of every streamer that appears in train interactions
train_items = set(train_with_negs["streamer_id"].unique())

# positives in the existing test split
test_pos = test_df.loc[test_df["label"] == 1, ["user_id", "streamer_id"]]

# cold positives = streamer_id not seen in train
cold_pos = test_pos[~test_pos["streamer_id"].isin(train_items)]
cold_item_set = set(cold_pos["streamer_id"])
print("cold positives:", len(cold_pos), "unique cold items:",
      len(cold_item_set))


cold positives: 1 unique cold items: 1


In [93]:
# keep only the user groups that have a cold positive
cold_users = set(cold_pos["user_id"])

# filter the whole test_df to those users only
cold_test = test_df[test_df["user_id"].isin(cold_users)].copy()

In [94]:
cold_test

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z,score
67872,1237655,6018912,1,0.0,1,456.0,936.0,0.0,351637.540935,93.836723,93.836723,25.102483,14.900157,6.144314e+05,3276.324583,0.000002,-0.283372
67873,1237655,5851092,0,0.0,1,456.0,936.0,0.0,235929.000000,65.000000,65.000000,27.000000,8.000000,1.176109e+06,3629.676923,-0.301603,-0.191841
67874,1237655,2556187,0,0.0,1,456.0,936.0,0.0,93167.000000,35.000000,35.000000,8.000000,0.000000,8.961600e+04,2661.914286,-0.615375,-0.829337
67875,1237655,1839607,0,0.0,1,456.0,936.0,0.0,770586.000000,190.000000,190.000000,44.000000,23.000000,8.311650e+05,4055.715789,1.005781,0.208151
67876,1237655,1823675,0,0.0,1,456.0,936.0,0.0,192.000000,2.000000,2.000000,1.000000,0.000000,4.200000e+01,96.000000,-0.960525,-3.112854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67968,1237655,6135019,0,0.0,1,456.0,936.0,0.0,351637.540935,93.836723,93.836723,25.102483,14.900157,6.144314e+05,3276.324583,0.000002,-0.283372
67969,1237655,4773834,0,0.0,1,456.0,936.0,0.0,58981.000000,151.000000,151.000000,14.000000,40.000000,7.941100e+04,390.602649,0.597877,-0.524362
67970,1237655,1468670,0,0.0,1,456.0,936.0,0.0,10310.000000,32.000000,32.000000,42.000000,2.000000,2.018900e+04,322.187500,-0.646753,-1.169171
67971,1237655,6751506,0,0.0,1,456.0,936.0,0.0,76712.000000,68.000000,68.000000,11.000000,6.000000,4.058100e+04,1128.117647,-0.270226,-0.518100


In [95]:
X_cold_test = scaler.transform(cold_test[FEATURE_COLS])
y_cold_test = cold_test["label"].values

scores = ranker.predict(X_cold_test)
cold_test["score"] = scores

sizes = cold_test.groupby("user_id").size().to_numpy()  # group vector
idx = np.cumsum(sizes)[:-1]  # cumulative sum, indicating where each group ends
groups = np.split(cold_test.to_numpy(), idx)  # list of arrays

ranks = []  # rank (1-based) of the positive per user
for group in groups:
    # sort descending by score inside this user group
    group_sorted = group[np.argsort(-group[:, SCORE_COL])]
    # index of the unique positive (label=1) in the sorted group
    pos_rank = np.where(group_sorted[:, LABEL_COL] == 1)[0][0] + 1
    ranks.append(pos_rank)  # store the rank (1-based)
ranks = np.asarray(ranks)  # shape [n_users]

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [96]:
KS = (10, 20, 50)
for k in KS:
    print(f"k={k:<2}  "
          f"Recall(Hit rate) {recall_at_k(ranks,k):.4f}  "
          f"nDCG {ndcg_at_k(ranks,k):.4f}  "
          f"MRR  {mrr_at_k(ranks,k):.4f}")

k=10  Recall(Hit rate) 0.0000  nDCG 0.0000  MRR  0.0000
k=20  Recall(Hit rate) 0.0000  nDCG 0.0000  MRR  0.0000
k=50  Recall(Hit rate) 1.0000  nDCG 0.1854  MRR  0.0244


# Test

In [29]:
import pandas as pd
import numpy as np
import argparse
import pathlib
import json
from lightgbm import Booster
from joblib import load

USER_ID_COL = "user_id"
STREAMER_ID_COL = "streamer_id"

DEFAULT_K = 20 # Default number of candidates to retrieve

FEATURE_COLS = [
    'u_watch_tot', 'u_watch_cnt', 'u_gift_cnt', 'u_gift_amt', 'u_follow_cnt',
    'i_watch_tot', 'i_watch_cnt', 'i_unique_user', 'i_live_cnt',
    'i_followers', 'i_gift_amt', 'i_watch_avg', 'i_pop_z'
]

In [2]:
user_id = 1000015
retrieval_path = f"/Users/tu/Documents/lang_recsys/data/retrieval_results/user_{user_id}.json"
feature_dir = "/Users/tu/Documents/lang_recsys/features/ranker"


In [10]:
# Load retrieval result
print(f"› Loading retrieved candidates from {retrieval_path}")
with open(retrieval_path, "r", encoding="utf-8") as f:
    recs = json.load(f)

candidates_df = pd.DataFrame(recs)

› Loading retrieved candidates from /Users/tu/Documents/lang_recsys/data/retrieval_results/user_1000015.json


In [25]:
candidates_df.columns

Index(['streamer_id', 'score'], dtype='object')

In [11]:
user_feats = pd.read_parquet(f"{feature_dir}/user.parquet")
item_feats = pd.read_parquet(f"{feature_dir}/item.parquet")

In [26]:
def load_features(user_id: int, candidates: pd.DataFrame, user_feats: pd.DataFrame, item_feats: pd.DataFrame) -> pd.DataFrame:
    candidates = candidates.copy()
    candidates[USER_ID_COL] = user_id

    user_feats = user_feats.reset_index()
    item_feats = item_feats.reset_index()

    # Join user/item features
    df = candidates.merge(user_feats, on="user_id", how="left")
    df = df.merge(item_feats, on="streamer_id", how="left")
    df.fillna(df.mean(numeric_only=True), inplace=True)
    
    return df

In [27]:
df = load_features(user_id, candidates_df, user_feats, item_feats)
df.shape

(100, 16)

,streamer_id,score,user_id,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z
0,6596718,12.457895,1000015,0.0,14,534.0,782972.0,5.0,1075392.0,289,289,7.0,30.0,1731348.0,3721.079585,2.041859
1,6712401,11.925220,1000015,0.0,14,534.0,782972.0,5.0,1946407.0,256,256,19.0,23.0,6541286.0,7603.152344,1.696603
2,4485620,10.844104,1000015,0.0,14,534.0,782972.0,5.0,914548.0,239,239,71.0,49.0,1426504.0,3826.560669,1.518744
3,6759387,10.816029,1000015,0.0,14,534.0,782972.0,5.0,1359696.0,394,394,10.0,32.0,3129753.0,3451.005076,3.140400
4,6720569,10.781869,1000015,0.0,14,534.0,782972.0,5.0,823661.0,378,378,9.0,34.0,2837888.0,2178.997354,2.973004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,6603188,6.417753,1000015,0.0,14,534.0,782972.0,5.0,917606.0,341,341,46.0,84.0,371245.0,2690.926686,2.585899
96,2884788,6.410951,1000015,0.0,14,534.0,782972.0,5.0,2418156.0,181,181,52.0,21.0,809896.0,13359.977901,0.911930
97,6419241,6.405548,1000015,0.0,14,534.0,782972.0,5.0,1044286.0,113,113,64.0,21.0,6662144.0,9241.469027,0.200494
98,2940920,6.395640,1000015,0.0,14,534.0,782972.0,5.0,1443796.0,182,182,54.0,26.0,1929837.0,7932.945055,0.922393


In [30]:
test_path = "/Users/tu/Documents/lang_recsys/data/splits/donate/test.parquet"

test_df = pd.read_parquet(test_path) if test_path.endswith(".parquet") else pd.read_csv(args.test_path)
user_ids = test_df["user_id"].unique()

In [32]:
len(user_ids)

12850